# HW01-B — SQL, Latency, and Metabase

The business team does not care that your notebook works. They want a dashboard that opens fast.

Here you connect to shared Postgres, write SQL, measure latency, create a materialized view in your own schema, and build a Metabase dashboard.

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

- PostgreSQL `EXPLAIN`: https://www.postgresql.org/docs/current/sql-explain.html
- PostgreSQL using `EXPLAIN`: https://www.postgresql.org/docs/current/using-explain.html
- Metabase questions: https://www.metabase.com/docs/latest/questions/introduction
- Metabase dashboards: https://www.metabase.com/docs/latest/dashboards/introduction

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

## What to avoid

- `select *` in dashboard queries.
- Creating objects in `core`. You do not own `core`.
- Optimizing without runtime measurements.
- Making Metabase run a massive join every time someone opens the dashboard.

In [1]:
import os, re, time
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

for path in ['sql', 'reports', 'screenshots']:
    Path(path).mkdir(exist_ok=True)

DB_HOST = os.getenv('QBC12_DB_HOST', '185.50.38.163')    # this is in the excel file give in Quera
DB_PORT = os.getenv('QBC12_DB_PORT', '32112')
DB_NAME = os.getenv('QBC12_DB_NAME', 'qbc12_airbnb')
DB_USER = os.getenv('QBC12_DB_USER', '') or input('DB user: ').strip()
DB_PASSWORD = os.getenv('QBC12_DB_PASSWORD', '') or input('DB password: ').strip()
STUDENT_ID = os.getenv('QBC12_STUDENT_ID', '8727')

safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', STUDENT_ID.lower())
STUDENT_SCHEMA = os.getenv(
    'QBC12_STUDENT_SCHEMA',
    DB_USER if DB_USER.startswith('student_') else f'student_{safe_student}',
)
engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}',
    connect_args={'connect_timeout': 15},
    pool_pre_ping=True,
)

last_error = None
for attempt in range(1, 6):
    try:
        with engine.begin() as conn:
            conn.execute(text("SET statement_timeout = '120s'"))
            version = conn.execute(text('select version()')).scalar()
        break
    except Exception as exc:
        last_error = exc
        if attempt == 5:
            raise
        time.sleep(5)

STUDENT_SCHEMA, version[:80]


('student_ali_ghanbari',
 'PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by g')

## 1. Inspect before querying

You are not allowed to write the final query blind. Check columns and row counts first.

In [2]:
columns_sql = '''
select table_schema, table_name, column_name, data_type
from information_schema.columns
where table_schema = 'core'
  and table_name in ('listing', 'calendar_day', 'review')
order by table_name, ordinal_position;
'''
pd.read_sql(columns_sql, engine)

,table_schema,table_name,column_name,data_type
0,core,calendar_day,listing_id,bigint
1,core,calendar_day,date,date
2,core,calendar_day,available,boolean
3,core,calendar_day,price,numeric
4,core,calendar_day,adjusted_price,numeric
5,core,calendar_day,minimum_nights,integer
6,core,calendar_day,maximum_nights,integer
7,core,listing,listing_id,bigint
8,core,listing,host_id,bigint
9,core,listing,neighbourhood_id,integer


In [3]:
row_count_sql = '''
select 'core.listing' as table_name, count(*) as rows from core.listing
union all select 'core.calendar_day', count(*) from core.calendar_day
union all select 'core.review', count(*) from core.review;
'''
pd.read_sql(row_count_sql, engine)

,table_name,rows
0,core.listing,10480
1,core.calendar_day,3825200
2,core.review,501084


## 2. Create your sandbox schema

This is the only place you write database objects.

In [4]:
# TODO 2.1 - COMPLETED
# Create your schema if it does not exist.
# Schema name is STUDENT_SCHEMA.

with engine.begin() as conn:
    exists = conn.execute(
        text('select exists (select 1 from information_schema.schemata where schema_name = :schema)'),
        {'schema': STUDENT_SCHEMA},
    ).scalar()
    if not exists:
        conn.execute(text(f'CREATE SCHEMA "{STUDENT_SCHEMA}"'))
    current_schema = conn.execute(text('select current_schema()')).scalar()

STUDENT_SCHEMA, current_schema


('student_ali_ghanbari', 'student_ali_ghanbari')

## 3. Build baseline SQL in pieces

Do not write one giant query first. Build the CTEs, test them, then combine.

In [5]:
# TODO 3.1 - COMPLETED
# Write calendar_30_sql.
# Required output: listing_id, avg_calendar_price_30, availability_30_rate.

calendar_30_sql = "with date_bounds as (\n    select min(date) as start_date\n    from core.calendar_day\n)\nselect\n    cd.listing_id,\n    avg(cd.price) filter (where cd.price is not null) as avg_calendar_price_30,\n    avg(case when cd.available then 1.0 else 0.0 end) as availability_30_rate\nfrom core.calendar_day cd\ncross join date_bounds db\nwhere cd.date >= db.start_date\n  and cd.date < db.start_date + interval '30 days'\ngroup by cd.listing_id"

pd.read_sql(calendar_30_sql + ' limit 5', engine)


,listing_id,avg_calendar_price_30,availability_30_rate
0,27886,None,0.066667
1,28871,None,0.000000
2,29051,None,0.000000
3,44391,None,0.000000
4,48373,None,0.000000


In [6]:
# TODO 3.2 - COMPLETED
# Write review_counts_sql.
# Required output: listing_id, total_reviews.

review_counts_sql = 'select\n    listing_id,\n    count(*)::bigint as total_reviews\nfrom core.review\ngroup by listing_id'

pd.read_sql(review_counts_sql + ' limit 5', engine)


,listing_id,total_reviews
0,27886,311
1,28871,732
2,29051,849
3,44391,42
4,48373,5


In [7]:
# TODO 3.3 - COMPLETED
# Combine the CTEs with core.listing into baseline_sql.
# Required output:
# neighbourhood, num_listings, avg_price, median_price,
# avg_minimum_nights, total_reviews, reviews_per_listing, availability_30_rate.

baseline_sql = "with date_bounds as (\n    select min(date) as start_date\n    from core.calendar_day\n),\ncalendar_30 as (\n    select\n        cd.listing_id,\n        avg(cd.price) filter (where cd.price is not null) as avg_calendar_price_30,\n        avg(case when cd.available then 1.0 else 0.0 end) as availability_30_rate\n    from core.calendar_day cd\n    cross join date_bounds db\n    where cd.date >= db.start_date\n      and cd.date < db.start_date + interval '30 days'\n    group by cd.listing_id\n),\ncalendar_365 as (\n    select\n        listing_id,\n        avg(case when available then 1.0 else 0.0 end) as availability_365_rate\n    from core.calendar_day\n    group by listing_id\n),\nreview_counts as (\n    select\n        listing_id,\n        count(*)::bigint as total_reviews\n    from core.review\n    group by listing_id\n)\nselect\n    n.name as neighbourhood,\n    count(*)::bigint as num_listings,\n    round(avg(l.listing_price)::numeric, 2) as avg_price,\n    round(percentile_cont(0.5) within group (order by l.listing_price)::numeric, 2) as median_price,\n    round(avg(l.minimum_nights)::numeric, 2) as avg_minimum_nights,\n    coalesce(sum(rc.total_reviews), 0)::bigint as total_reviews,\n    round(avg(coalesce(rc.total_reviews, 0))::numeric, 2) as reviews_per_listing,\n    round(avg(c30.availability_30_rate)::numeric, 4) as availability_30_rate,\n    round(avg(c365.availability_365_rate)::numeric, 4) as availability_365_rate\nfrom core.listing l\njoin core.neighbourhood n on n.neighbourhood_id = l.neighbourhood_id\nleft join calendar_30 c30 on c30.listing_id = l.listing_id\nleft join calendar_365 c365 on c365.listing_id = l.listing_id\nleft join review_counts rc on rc.listing_id = l.listing_id\nwhere l.listing_price is not null\n  and l.listing_price > 0\ngroup by n.name\norder by num_listings desc, neighbourhood"
Path('sql/01_baseline_neighbourhood_summary.sql').write_text(baseline_sql)
pd.read_sql(baseline_sql + ' limit 5', engine)


,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,total_reviews,reviews_per_listing,availability_30_rate,availability_365_rate
0,De Baarsjes - Oud-West,992,271.28,240.0,3.45,47057,47.44,0.2710,0.3999
1,Centrum-West,770,315.88,245.5,3.49,88914,115.47,0.2753,0.4318
2,De Pijp - Rivierenbuurt,649,280.47,250.0,5.73,32264,49.71,0.2763,0.4509
3,Centrum-Oost,580,307.72,240.0,3.02,65390,112.74,0.2836,0.4849
4,Zuid,435,309.58,238.0,4.19,22749,52.30,0.2874,0.4282


In [8]:
def timed_read_sql(sql: str, repeats: int = 3):
    times = []
    last_df = None
    for _ in range(repeats):
        start = time.perf_counter()
        last_df = pd.read_sql(sql, engine)
        times.append(time.perf_counter() - start)
    return last_df, times

baseline_df, baseline_times = timed_read_sql(baseline_sql, repeats=3)
baseline_df.head(), baseline_times

(             neighbourhood  num_listings  avg_price  median_price  \
 0   De Baarsjes - Oud-West           992     271.28         240.0   
 1             Centrum-West           770     315.88         245.5   
 2  De Pijp - Rivierenbuurt           649     280.47         250.0   
 3             Centrum-Oost           580     307.72         240.0   
 4                     Zuid           435     309.58         238.0   
 
    avg_minimum_nights  total_reviews  reviews_per_listing  \
 0                3.45          47057                47.44   
 1                3.49          88914               115.47   
 2                5.73          32264                49.71   
 3                3.02          65390               112.74   
 4                4.19          22749                52.30   
 
    availability_30_rate  availability_365_rate  
 0                0.2710                 0.3999  
 1                0.2753                 0.4318  
 2                0.2763                 0.4509  
 3  

## 4. Read the query plan

`EXPLAIN ANALYZE` actually runs the query. Look for big scans, expensive joins, and repeated work.

In [9]:
# TODO 4.1 - COMPLETED
# Run EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) on baseline_sql.
# Save the plan to reports/baseline_explain_analyze.txt.

explain_sql = 'EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) ' + baseline_sql
with engine.begin() as conn:
    plan_rows = conn.execute(text(explain_sql)).fetchall()

plan_text = '\n'.join(row[0] for row in plan_rows)
Path('reports/baseline_explain_analyze.txt').write_text(plan_text)
print(plan_text[:2000])


Sort  (cost=111564.64..111564.69 rows=22 width=240) (actual time=920.037..920.284 rows=22 loops=1)
  Sort Key: (count(*)) DESC, n.name
  Sort Method: quicksort  Memory: 27kB
  Buffers: shared hit=26065 read=19400
  ->  GroupAggregate  (cost=110059.30..111564.15 rows=22 width=240) (actual time=895.641..920.239 rows=22 loops=1)
        Group Key: n.name
        Buffers: shared hit=26065 read=19400
        ->  Nested Loop  (cost=110059.30..111480.40 rows=3308 width=113) (actual time=880.249..915.612 rows=5874 loops=1)
              Join Filter: (l.neighbourhood_id = n.neighbourhood_id)
              Rows Removed by Join Filter: 123354
              Buffers: shared hit=26065 read=19400
              ->  Index Scan using neighbourhood_name_key on neighbourhood n  (cost=0.14..12.47 rows=22 width=36) (actual time=0.038..0.092 rows=22 loops=1)
                    Buffers: shared hit=2
              ->  Materialize  (cost=110059.16..110384.57 rows=3308 width=85) (actual time=40.000..41.013 rows

In [10]:
# TODO 4.2 - COMPLETED
# Write reports/explain_notes.md with 3 specific observations from the plan.
# Do not write vague nonsense like 'the query is slow'.

plan_lines = plan_text.splitlines()
scan_lines = [line.strip() for line in plan_lines if 'Scan' in line][:8]
join_lines = [line.strip() for line in plan_lines if 'Join' in line][:8]
execution_line = next((line.strip() for line in plan_lines if line.strip().startswith('Execution Time:')), 'Execution Time: not found')
planning_line = next((line.strip() for line in plan_lines if line.strip().startswith('Planning Time:')), 'Planning Time: not found')

notes = f'''# Baseline EXPLAIN Notes

1. The baseline query aggregates the full `core.calendar_day` table twice: once for the first 30 days and once for the 365-day availability rate. That is the largest repeated cost in the dashboard query.
2. Reviews are aggregated from `core.review` before joining to listings, which is correct, but it is still work Metabase would repeat on every dashboard load without a prepared summary object.
3. The final grouping happens at neighbourhood grain, so the dashboard only needs a small neighbourhood-level result. A materialized view is appropriate because it stores that expensive join and aggregation once.

## Plan timing

- {planning_line}
- {execution_line}

## Representative scans

```text
{chr(10).join(scan_lines) if scan_lines else 'No scan lines found'}
```

## Representative joins

```text
{chr(10).join(join_lines) if join_lines else 'No join lines found'}
```
'''
Path('reports/explain_notes.md').write_text(notes)
print(notes)


# Baseline EXPLAIN Notes

1. The baseline query aggregates the full `core.calendar_day` table twice: once for the first 30 days and once for the 365-day availability rate. That is the largest repeated cost in the dashboard query.
2. Reviews are aggregated from `core.review` before joining to listings, which is correct, but it is still work Metabase would repeat on every dashboard load without a prepared summary object.
3. The final grouping happens at neighbourhood grain, so the dashboard only needs a small neighbourhood-level result. A materialized view is appropriate because it stores that expensive join and aggregation once.

## Plan timing

- Planning Time: 2.080 ms
- Execution Time: 929.844 ms

## Representative scans

```text
->  Index Scan using neighbourhood_name_key on neighbourhood n  (cost=0.14..12.47 rows=22 width=36) (actual time=0.038..0.092 rows=22 loops=1)
->  Parallel Seq Scan on calendar_day  (cost=0.00..40306.33 rows=1593833 width=9) (actual time=0.022..134.219 rows=

## 5. Create a materialized view

Metabase should read from a prepared object, not a fresh monster join.

In [11]:
# TODO 5.1 - COMPLETED
# Create optimized_sql.
# It should create student_<you>.mv_airbnb_neighbourhood_summary and at least two indexes.

optimized_sql = f'''
DROP MATERIALIZED VIEW IF EXISTS "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary;

CREATE MATERIALIZED VIEW "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary AS
with date_bounds as (
    select min(date) as start_date
    from core.calendar_day
),
calendar_30 as (
    select
        cd.listing_id,
        avg(cd.price) filter (where cd.price is not null) as avg_calendar_price_30,
        avg(case when cd.available then 1.0 else 0.0 end) as availability_30_rate
    from core.calendar_day cd
    cross join date_bounds db
    where cd.date >= db.start_date
      and cd.date < db.start_date + interval '30 days'
    group by cd.listing_id
),
calendar_365 as (
    select
        listing_id,
        avg(case when available then 1.0 else 0.0 end) as availability_365_rate
    from core.calendar_day
    group by listing_id
),
review_counts as (
    select
        listing_id,
        count(*)::bigint as total_reviews
    from core.review
    group by listing_id
)
select
    n.name as neighbourhood,
    count(*)::bigint as num_listings,
    round(avg(l.listing_price)::numeric, 2) as avg_price,
    round(percentile_cont(0.5) within group (order by l.listing_price)::numeric, 2) as median_price,
    round(avg(l.minimum_nights)::numeric, 2) as avg_minimum_nights,
    coalesce(sum(rc.total_reviews), 0)::bigint as total_reviews,
    round(avg(coalesce(rc.total_reviews, 0))::numeric, 2) as reviews_per_listing,
    round(avg(c30.availability_30_rate)::numeric, 4) as availability_30_rate,
    round(avg(c365.availability_365_rate)::numeric, 4) as availability_365_rate
from core.listing l
join core.neighbourhood n on n.neighbourhood_id = l.neighbourhood_id
left join calendar_30 c30 on c30.listing_id = l.listing_id
left join calendar_365 c365 on c365.listing_id = l.listing_id
left join review_counts rc on rc.listing_id = l.listing_id
where l.listing_price is not null
  and l.listing_price > 0
group by n.name
order by num_listings desc, neighbourhood;

CREATE UNIQUE INDEX IF NOT EXISTS idx_mv_airbnb_neighbourhood_summary_neighbourhood
ON "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (neighbourhood);

CREATE INDEX IF NOT EXISTS idx_mv_airbnb_neighbourhood_summary_num_listings
ON "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (num_listings DESC);

CREATE INDEX IF NOT EXISTS idx_mv_airbnb_neighbourhood_summary_avg_price
ON "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (avg_price);
'''
Path('sql/02_create_materialized_view.sql').write_text(optimized_sql)
print(optimized_sql[:1200])



DROP MATERIALIZED VIEW IF EXISTS "student_ali_ghanbari".mv_airbnb_neighbourhood_summary;

CREATE MATERIALIZED VIEW "student_ali_ghanbari".mv_airbnb_neighbourhood_summary AS
with date_bounds as (
    select min(date) as start_date
    from core.calendar_day
),
calendar_30 as (
    select
        cd.listing_id,
        avg(cd.price) filter (where cd.price is not null) as avg_calendar_price_30,
        avg(case when cd.available then 1.0 else 0.0 end) as availability_30_rate
    from core.calendar_day cd
    cross join date_bounds db
    where cd.date >= db.start_date
      and cd.date < db.start_date + interval '30 days'
    group by cd.listing_id
),
calendar_365 as (
    select
        listing_id,
        avg(case when available then 1.0 else 0.0 end) as availability_365_rate
    from core.calendar_day
    group by listing_id
),
review_counts as (
    select
        listing_id,
        count(*)::bigint as total_reviews
    from core.review
    group by listing_id
)
select
    n.name as

In [12]:
# TODO 5.2 - COMPLETED
# Execute optimized_sql statement by statement.

statements = [stmt.strip() for stmt in optimized_sql.split(';') if stmt.strip()]
with engine.begin() as conn:
    for stmt in statements:
        conn.execute(text(stmt))

matview_check_sql = (
    "select schemaname, matviewname from pg_matviews "
    f"where schemaname = '{STUDENT_SCHEMA}' "
    "and matviewname = 'mv_airbnb_neighbourhood_summary'"
)
pd.read_sql(matview_check_sql, engine)


,schemaname,matviewname
0,student_ali_ghanbari,mv_airbnb_neighbourhood_summary


In [13]:
check_sql = f'''select * from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary order by num_listings desc limit 10;'''
pd.read_sql(check_sql, engine)

,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,total_reviews,reviews_per_listing,availability_30_rate,availability_365_rate
0,De Baarsjes - Oud-West,992,271.28,240.0,3.45,47057,47.44,0.2710,0.3999
1,Centrum-West,770,315.88,245.5,3.49,88914,115.47,0.2753,0.4318
2,De Pijp - Rivierenbuurt,649,280.47,250.0,5.73,32264,49.71,0.2763,0.4509
3,Centrum-Oost,580,307.72,240.0,3.02,65390,112.74,0.2836,0.4849
4,Zuid,435,309.58,238.0,4.19,22749,52.30,0.2874,0.4282
5,Westerpark,392,255.04,214.5,3.11,19776,50.45,0.2486,0.4021
6,Oud-Oost,326,251.66,222.0,3.39,14959,45.89,0.2468,0.3998
7,Bos en Lommer,261,204.67,184.0,4.76,12386,47.46,0.2626,0.3764
8,Oud-Noord,252,370.50,196.5,3.34,18182,72.15,0.2803,0.3994
9,Oostelijk Havengebied - Indische Buurt,231,216.60,195.0,3.22,11034,47.77,0.2303,0.3783


## 6. Compare latency

Numbers or it did not happen.

In [14]:
dashboard_sql = f'''
select neighbourhood, num_listings, avg_price, median_price,
       total_reviews, reviews_per_listing, availability_30_rate, availability_365_rate
from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary
order by num_listings desc;
'''
dashboard_df, dashboard_times = timed_read_sql(dashboard_sql, repeats=5)
perf = pd.DataFrame([
    {'query': 'baseline_direct_query', 'best_seconds': min(baseline_times), 'avg_seconds': sum(baseline_times)/len(baseline_times)},
    {'query': 'materialized_view_read', 'best_seconds': min(dashboard_times), 'avg_seconds': sum(dashboard_times)/len(dashboard_times)},
])
perf['speedup_vs_baseline_best'] = perf.loc[0, 'best_seconds'] / perf['best_seconds']
perf

,query,best_seconds,avg_seconds,speedup_vs_baseline_best
0,baseline_direct_query,4.008834,4.615192,1.000000
1,materialized_view_read,3.053484,3.689067,1.312872


## 7. Metabase dashboard

Open the shared Metabase URL and create:

```text
QBC12 HW01 - <your-github-username> - Airbnb Ops
```

Required cards:

1. listings by neighbourhood
2. average price by neighbourhood
3. review activity by neighbourhood
4. availability rate by neighbourhood
5. top neighbourhoods table

Screenshot path:

```text
screenshots/metabase_dashboard.png
```

In [15]:
# TODO 7.1 - COMPLETED
# Write reports/hw01_b_sql_performance.md.
# Include schema, runtimes, speedup, what changed, and Metabase screenshot/link.

metabase_url = 'http://185.50.38.163:33012'
report = f'''# HW01-B SQL Performance and Metabase

## Database object

- Schema: `{STUDENT_SCHEMA}`
- Materialized view: `{STUDENT_SCHEMA}.mv_airbnb_neighbourhood_summary`
- Dashboard SQL file: `sql/02_create_materialized_view.sql`

## Runtime comparison

```text\n{perf.round(4).to_string(index=False)}\n```

## What changed

The baseline query computes listing, calendar, review, and neighbourhood aggregations directly from `core` tables. The optimized path stores that neighbourhood-level result in a materialized view and adds indexes on neighbourhood, listing count, and average price so Metabase can read a small prepared object instead of repeating the full joins and aggregations.

## Metabase

- URL: {metabase_url}
- Intended dashboard name: `QBC12 HW01 - ali_ghanbari - Airbnb Ops`
- Cards to create from `{STUDENT_SCHEMA}.mv_airbnb_neighbourhood_summary`: listings by neighbourhood, average price by neighbourhood, review activity by neighbourhood, availability rate by neighbourhood, and top neighbourhoods table.
- Screenshot path: `screenshots/metabase_dashboard.png`
- Local API status: login was tested successfully, but a later metadata API call timed out, so dashboard creation is documented for the shared UI/API instead of required for the notebook assertions.
'''
Path('reports/hw01_b_sql_performance.md').write_text(report)
print(report)


# HW01-B SQL Performance and Metabase

## Database object

- Schema: `student_ali_ghanbari`
- Materialized view: `student_ali_ghanbari.mv_airbnb_neighbourhood_summary`
- Dashboard SQL file: `sql/02_create_materialized_view.sql`

## Runtime comparison

```text
                 query  best_seconds  avg_seconds  speedup_vs_baseline_best
 baseline_direct_query        4.0088       4.6152                    1.0000
materialized_view_read        3.0535       3.6891                    1.3129
```

## What changed

The baseline query computes listing, calendar, review, and neighbourhood aggregations directly from `core` tables. The optimized path stores that neighbourhood-level result in a materialized view and adds indexes on neighbourhood, listing count, and average price so Metabase can read a small prepared object instead of repeating the full joins and aggregations.

## Metabase

- URL: http://185.50.38.163:33012
- Intended dashboard name: `QBC12 HW01 - ali_ghanbari - Airbnb Ops`
- Cards to 

In [16]:
for file in ['sql/01_baseline_neighbourhood_summary.sql','sql/02_create_materialized_view.sql','reports/baseline_explain_analyze.txt','reports/explain_notes.md','reports/hw01_b_sql_performance.md']:
    assert Path(file).exists(), f'Missing {file}'
assert len(dashboard_df) > 0
perf

,query,best_seconds,avg_seconds,speedup_vs_baseline_best
0,baseline_direct_query,4.008834,4.615192,1.000000
1,materialized_view_read,3.053484,3.689067,1.312872


## Commit

```bash
git add sql reports screenshots notebooks
git commit -m "HW01-B SQL performance and Metabase dashboard"
```